# V1DD 1196 CCF coordinates

Transform native CAVE nucleus positions to CCFv3 micrometers and save them as CCM spatial locations.

Run [etl_v1dd1196_01.ipynb](etl_v1dd1196_01.ipynb) first, using the same configured output root. This notebook retains only its registered single-soma neuron cohort; it does not register additional cells or change existing CAVE or streamline coordinates.

Input: `data/EM v1dd 1196/soma_and_cell_type_1196.feather`. The native `pt_position_x/y/z` columns are in nanometers.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl

from connects_common_connectivity.config import get_settings
from connects_common_connectivity.io import (
    read_reference_spaces,
    read_spatial_locations,
    write_models,
)
from connects_common_connectivity.io.path_spec import MODEL_TABLE_PATHS
from connects_common_connectivity.models import (
    AnatomicalOrientation,
    Default2DView,
    LocationType,
    ReferenceSpace,
    SignedAxis,
    SpatialLocation,
    Unit,
)

In [ ]:
REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
DATA_ROOT = REPO_ROOT / "data/EM v1dd 1196"
OUTPUT_ROOT = Path(get_settings().output_root)

PROJECT_ID = "v1dd_1196"
DATASET_SINGLE_SOMA_NEURONS = "v1dd_1196_single_soma_neurons"
CCF_REFERENCE_SPACE_ID = "ccfv3_um"
POSITION_PATH = DATA_ROOT / "soma_and_cell_type_1196.feather"
COORDINATE_COLUMNS = ["pt_position_x", "pt_position_y", "pt_position_z"]

assert POSITION_PATH.is_file(), f"Missing local input: {POSITION_PATH}"
missing_paths = [
    OUTPUT_ROOT / MODEL_TABLE_PATHS[table]
    for table in ("DataItem", "DataItemDataSetAssociation")
    if not (OUTPUT_ROOT / MODEL_TABLE_PATHS[table]).is_dir()
]
assert not missing_paths, (
    f"Missing CCM tables: {missing_paths}. Run etl_v1dd1196_01.ipynb first."
)

## registered cells and native positions

Use the existing dataset associations as the cohort authority, preserving the base ETL's neuron selection and identity cleanup. Root IDs remain strings, without conversion through floating point. Require exactly one nucleus position for every registered cell; missing or ambiguous coordinates stop the ETL before writing.

In [ ]:
associations = pl.read_delta(
    OUTPUT_ROOT / MODEL_TABLE_PATHS["DataItemDataSetAssociation"]
).filter(
    (pl.col("project_id") == PROJECT_ID)
    & (pl.col("dataset_id") == DATASET_SINGLE_SOMA_NEURONS)
)
assert associations.height > 0, (
    "No V1DD 1196 cohort found; run etl_v1dd1196_01.ipynb first."
)
assert associations["dataitem_id"].null_count() == 0, "Null cohort cell IDs."
assert associations["dataitem_id"].n_unique() == associations.height, (
    "Duplicate cohort associations."
)
registered_ids = set(associations["dataitem_id"].to_list())
dataitems = pl.read_delta(OUTPUT_ROOT / MODEL_TABLE_PATHS["DataItem"]).filter(
    pl.col("project_id") == PROJECT_ID
)
assert registered_ids <= set(dataitems["id"].to_list()), (
    "Cohort associations reference unregistered DataItems."
)

positions = pd.read_feather(
    POSITION_PATH, columns=["id", "pt_root_id", *COORDINATE_COLUMNS]
)
assert pd.api.types.is_integer_dtype(positions["pt_root_id"].dtype), (
    "Expected integer root IDs; floating-point IDs may have lost precision."
)
positions["dataitem_id"] = positions["pt_root_id"].astype(str)
source_count = len(positions)
positions = positions.loc[
    positions["dataitem_id"].isin(registered_ids)
].sort_values("dataitem_id").reset_index(drop=True)
assert positions[["id", "pt_root_id"]].notna().all().all(), "Null nucleus/root IDs."
assert (positions["pt_root_id"] != 0).all(), "Invalid zero root ID."
assert positions["id"].is_unique, "Ambiguous nucleus IDs in the registered cohort."
assert positions["dataitem_id"].is_unique, "Multiple positions for one registered cell."
missing_ids = registered_ids - set(positions["dataitem_id"])
assert not missing_ids, f"Missing native positions for {len(missing_ids):,} registered cells."
assert positions[COORDINATE_COLUMNS].notna().all().all(), "Missing native coordinates."
xyz_nm = positions[COORDINATE_COLUMNS].to_numpy(dtype="float64")
assert np.isfinite(xyz_nm).all(), "Non-finite native coordinates."
print(f"source nuclei: {source_count:,}")
print(f"registered single-soma neurons retained: {len(positions):,}")
print(f"source nuclei outside the cohort: {source_count - len(positions):,}")

## CCF transform

The supplied 3-by-4 affine maps native CAVE **micrometers** to CCFv3 **micrometers**. Convert nanometers once, then apply `xyz_um @ M_V1DD[:, :3].T + M_V1DD[:, 3]`. The left block is rotation/scale and the last column is translation; there is no axis permutation, additional scaling, reflection, or clipping.

Increasing output X points anterior to posterior, Y dorsal to ventral, and Z subject left to right.

In [ ]:
M_V1DD = np.array([
    [-0.669823, -0.086371, -0.746876, 10050.8625],
    [-0.420765,  0.862503,  0.374974,   677.446],
    [-0.611796, -0.498627,  0.559825,  8785.9264],
])
assert M_V1DD.shape == (3, 4) and np.isfinite(M_V1DD).all()


def transform_to_ccf(xyz_nm):
    xyz_um = xyz_nm / 1_000
    return xyz_um @ M_V1DD[:, :3].T + M_V1DD[:, 3]


probe_nm = np.vstack([np.zeros(3), np.eye(3) * 1_000])
probe_ccf = transform_to_ccf(probe_nm)
np.testing.assert_allclose(probe_ccf[0], M_V1DD[:, 3], rtol=0, atol=1e-9)
np.testing.assert_allclose(
    probe_ccf[1:] - probe_ccf[0], M_V1DD[:, :3].T, rtol=0, atol=1e-9
)
ccf_um = transform_to_ccf(xyz_nm)
assert ccf_um.shape == (len(positions), 3), "Unexpected CCF coordinate shape."
assert np.isfinite(ccf_um).all(), "Non-finite CCF coordinates."
ccf_positions = pd.DataFrame(ccf_um, columns=["x", "y", "z"])
ccf_positions.insert(0, "dataitem_id", positions["dataitem_id"].to_numpy())
ccf_positions.head()

## spatial location

In [ ]:
orientation = AnatomicalOrientation(
    anterior_to_posterior=SignedAxis.PLUS_X,
    dorsal_to_ventral=SignedAxis.PLUS_Y,
    subject_left_to_right=SignedAxis.PLUS_Z,
)
reference_space = ReferenceSpace(
    id=CCF_REFERENCE_SPACE_ID,
    project_id=PROJECT_ID,
    name="CCFv3 micrometer coordinates",
    description=(
        "V1DD materialization 1196 native CAVE nucleus positions, converted "
        "from nanometers to micrometers and registered with M_V1DD in "
        "etl_v1dd1196_02_ccf_coord.ipynb."
    ),
    unit=Unit.MICRONS_LENGTH,
    default_2d_view=Default2DView(
        left_to_right=SignedAxis.PLUS_X,
        bottom_to_top=SignedAxis.MINUS_Y,
    ),
    anatomical_orientation=orientation,
)
spatial_locations = [
    SpatialLocation(
        project_id=PROJECT_ID,
        dataitem_id=row.dataitem_id,
        reference_space=CCF_REFERENCE_SPACE_ID,
        location_type=LocationType.SOMA,
        description="Soma nucleus position registered to CCFv3.",
        x=float(row.x), y=float(row.y), z=float(row.z),
    )
    for row in ccf_positions.itertuples(index=False)
]
print(write_models([reference_space], output_root=OUTPUT_ROOT))
print(write_models(spatial_locations, output_root=OUTPUT_ROOT))

## verification

Read back the project-specific metadata and SOMA coordinates, checking cohort coverage and numerical agreement. When configuration enables `dry_run`, writes are suppressed and read-back verification is explicitly skipped.

In [ ]:
if get_settings().dry_run:
    print("Dry run: no CCF tables written; stored-coordinate verification skipped.")
else:
    spaces = read_reference_spaces(
        project_id=PROJECT_ID,
        reference_space_ids=CCF_REFERENCE_SPACE_ID,
        output_root=OUTPUT_ROOT,
    )
    assert spaces.height == 1, "Expected one project-scoped CCF reference space."
    stored_space = spaces.row(0, named=True)
    assert stored_space["unit"] == Unit.MICRONS_LENGTH.value
    assert stored_space["anatomical_orientation"] is not None
    for direction, axis in orientation.model_dump(exclude_none=True).items():
        assert stored_space["anatomical_orientation"][direction] == axis
    stored = read_spatial_locations(
        PROJECT_ID,
        reference_spaces=CCF_REFERENCE_SPACE_ID,
        location_types=LocationType.SOMA,
        output_root=OUTPUT_ROOT,
    ).sort("dataitem_id")
    assert stored["dataitem_id"].to_list() == positions["dataitem_id"].tolist(), (
        "Stored CCF rows do not match the registered cohort exactly."
    )
    np.testing.assert_allclose(
        stored.select("x", "y", "z").to_numpy(), ccf_um, rtol=0, atol=1e-9
    )
    print(f"V1DD 1196 CCF SOMA coordinates verified: {stored.height:,}")